# SAVIA — Demo del DAO
## Sistema de Almacenamiento de Índices de Vegetación para Investigación Agronómica
### Proyecto Integrador Bases de Datos II · UNdeC 2026

Este notebook demuestra el funcionamiento del módulo `VinedoDAO` —
la capa de acceso a datos del sistema SAVIA.

In [13]:
from dao import VinedoDAO
from datetime import date
import matplotlib.pyplot as plt

dao = VinedoDAO()
print("Conexión establecida con MongoDB")
print(f"Base de datos: vinedos_chilecito")

Conexión establecida con MongoDB
Base de datos: vinedos_chilecito


In [14]:
# Consulta por zona
parcelas_nonogasta = dao.get_parcelas_por_zona("Nonogasta")
print(f"Parcelas en Nonogasta: {len(parcelas_nonogasta)}")
for p in parcelas_nonogasta:
    print(f"  - {p['nombre']} | {p['variedad']} | {p['superficie_ha']} ha")

# Consulta por cultivo
parcelas_vid = dao.get_parcelas_por_cultivo("vid")
print(f"\nTotal parcelas de vid: {len(parcelas_vid)}")
for p in parcelas_vid:
    print(f"  - {p['nombre']} ({p['zona']})")

Parcelas en Nonogasta: 2
  - Finca El Peñón | Torrontés Riojano | 3.8 ha
  - Finca El Peñón | Torrontés Riojano | 3.8 ha

Total parcelas de vid: 8
  - Finca El Peñón (Nonogasta)
  - Finca Los Sarmientos Norte (Los Sarmientos)
  - Viña Famatina (Famatina)
  - Finca Vichigasta Sur (Vichigasta)
  - Finca El Peñón (Nonogasta)
  - Finca Los Sarmientos Norte (Los Sarmientos)
  - Viña Famatina (Famatina)
  - Finca Vichigasta Sur (Vichigasta)


In [15]:
# Recuperar una parcela por id y agregar una observación nueva
parcela = parcelas_vid[0]
parcela_id = str(parcela["_id"])

print(f"Parcela seleccionada: {parcela['nombre']}")
print(f"ID: {parcela_id}")

# Agregar una observación nueva
nueva_obs = dao.agregar_observacion(
    parcela_id=parcela_id,
    observacion=__import__('db_models').Observacion(
        fecha=date(2024, 3, 18),
        ndvi=0.22,
        evi=0.16,
        ndwi=0.06,
        nubosidad_pct=2.1
    )
)
print(f"\nObservación agregada: {nueva_obs}")

Parcela seleccionada: Finca El Peñón
ID: 69fbc082dc94078a66858365

Observación agregada: True


In [16]:
# Consultar serie temporal completa
observaciones = dao.get_observaciones(parcela_id)
print(f"Total observaciones: {len(observaciones)}")
print("\nPrimeras 3:")
for o in observaciones[:3]:
    print(f"  {o['fecha']} | NDVI: {o['ndvi']} | EVI: {o['evi']} | Nubosidad: {o['nubosidad_pct']}%")

# Consultar por rango de fechas
obs_verano = dao.get_observaciones(
    parcela_id,
    desde=date(2023, 12, 1),
    hasta=date(2024, 2, 28)
)
print(f"\nObservaciones en verano (dic-feb): {len(obs_verano)}")

Total observaciones: 14

Primeras 3:
  2023-09-05 | NDVI: 0.31 | EVI: 0.22 | Nubosidad: 3.2%
  2023-09-20 | NDVI: 0.38 | EVI: 0.27 | Nubosidad: 7.1%
  2023-10-05 | NDVI: 0.45 | EVI: 0.33 | Nubosidad: 2.8%

Observaciones en verano (dic-feb): 6


In [17]:
fechas = [o["fecha"] for o in observaciones]
ndvi_vals = [o["ndvi"] for o in observaciones]
evi_vals = [o["evi"] for o in observaciones]

plt.figure(figsize=(12, 5))
plt.plot(fechas, ndvi_vals, marker="o", label="NDVI", color="#2d6a4f", linewidth=2)
plt.plot(fechas, evi_vals, marker="s", label="EVI", color="#74c69d", linewidth=2)
plt.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Umbral 0.5")
plt.title("Serie temporal de índices espectrales — Finca El Peñón (Nonogasta)")
plt.xlabel("Fecha")
plt.ylabel("Valor del índice")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [18]:
# NDVI promedio por zona para la temporada 2023/2024
resultado = dao.get_ndvi_promedio_por_zona("Nonogasta", "2023/2024")
for r in resultado:
    print(f"Zona: {r['_id']}")
    print(f"  NDVI promedio: {r['ndvi_promedio']:.4f}")
    print(f"  Total observaciones: {r['total_observaciones']}")

Zona: Nonogasta
  NDVI promedio: 0.5059
  Total observaciones: 27


In [19]:
# Campañas registradas
campanas = dao.get_campanas(parcela_id)
print(f"Historial de cosechas — Finca El Peñón")
print("-" * 45)
for c in campanas:
    print(f"Temporada: {c['temporada']}")
    print(f"  Cosecha:      {c['fecha_cosecha']}")
    print(f"  Rendimiento:  {c['rendimiento_kg_ha']} kg/ha")
    print(f"  Notas:        {c['notas']}")

dao.cerrar()
print("\nConexión cerrada.")

Historial de cosechas — Finca El Peñón
---------------------------------------------
Temporada: 2023/2024
  Cosecha:      2024-02-28
  Rendimiento:  8400 kg/ha
  Notas:        Sin eventos climáticos adversos. Riego controlado desde noviembre.

Conexión cerrada.


## Consultas Geoespaciales

Las siguientes celdas demuestran el uso del índice `2dsphere` sobre el campo `geometria` de la colección `parcelas`, declarado en `setup_db.py`.
Toda la lógica espacial la ejecuta MongoDB internamente.

In [20]:
from dao import VinedoDAO

dao = VinedoDAO()

In [21]:
# get_parcelas_cerca_de — $nearSphere
# Devuelve parcelas dentro de un radio dado desde un punto.
# Los resultados vienen ordenados por distancia, de más cercana a más lejana.

# Punto de referencia: centro aproximado del departamento Chilecito
# Las 4 parcelas del seed están entre 2 y 8 km de este punto

LAT_CHILECITO = -29.01
LON_CHILECITO = -67.49
RADIO_KM = 8

parcelas_cercanas = dao.get_parcelas_cerca_de(
    lat=LAT_CHILECITO,
    lon=LON_CHILECITO,
    radio_km=RADIO_KM
)

print(f"Parcelas dentro de {RADIO_KM} km del centro de Chilecito: {len(parcelas_cercanas)}\n")
for p in parcelas_cercanas:
    print(f"  {p['nombre']:30s}  zona={p['zona']:20s}  altitud={p.get('altitud_msnm', '—')} msnm")

Parcelas dentro de 8 km del centro de Chilecito: 8

  Finca El Peñón                  zona=Nonogasta             altitud=1050 msnm
  Finca El Peñón                  zona=Nonogasta             altitud=1050 msnm
  Finca Los Sarmientos Norte      zona=Los Sarmientos        altitud=980 msnm
  Finca Los Sarmientos Norte      zona=Los Sarmientos        altitud=980 msnm
  Viña Famatina                   zona=Famatina              altitud=1120 msnm
  Viña Famatina                   zona=Famatina              altitud=1120 msnm
  Finca Vichigasta Sur            zona=Vichigasta            altitud=920 msnm
  Finca Vichigasta Sur            zona=Vichigasta            altitud=920 msnm


In [22]:
# Reducir el radio para ver cómo cambia el resultado
# Nonogasta y Los Sarmientos están a ~2-3 km del centro,
# Famatina y Vichigasta quedan fuera con radio pequeño

parcelas_radio_chico = dao.get_parcelas_cerca_de(
    lat=LAT_CHILECITO,
    lon=LON_CHILECITO,
    radio_km=3
)

print(f"Con radio 3 km → {len(parcelas_radio_chico)} parcela(s):")
for p in parcelas_radio_chico:
    print(f"  {p['nombre']}")

Con radio 3 km → 4 parcela(s):
  Finca El Peñón
  Finca El Peñón
  Finca Los Sarmientos Norte
  Finca Los Sarmientos Norte


In [23]:
# get_parcelas_en_bbox — $geoWithin + $geometry
# Devuelve parcelas dentro de un rectángulo geográfico definido por esquinas SW y NE.
# Nota: $box no funciona con índice 2dsphere — $geoWithin + $geometry es la forma correcta.

# Bounding box amplio: cubre todas las zonas del seed
parcelas_en_area = dao.get_parcelas_en_bbox(
    sw_lat=-29.05, sw_lon=-67.53,
    ne_lat=-28.98, ne_lon=-67.45
)

print(f"Parcelas en el área completa del departamento: {len(parcelas_en_area)}\n")
for p in parcelas_en_area:
    print(f"  {p['nombre']:30s}  zona={p['zona']}")

Parcelas en el área completa del departamento: 8

  Viña Famatina                   zona=Famatina
  Viña Famatina                   zona=Famatina
  Finca El Peñón                  zona=Nonogasta
  Finca El Peñón                  zona=Nonogasta
  Finca Los Sarmientos Norte      zona=Los Sarmientos
  Finca Los Sarmientos Norte      zona=Los Sarmientos
  Finca Vichigasta Sur            zona=Vichigasta
  Finca Vichigasta Sur            zona=Vichigasta


In [24]:
# Bbox reducido: solo el sector norte (Famatina)
# lat > -29.00 excluye Nonogasta, Los Sarmientos y Vichigasta

parcelas_norte = dao.get_parcelas_en_bbox(
    sw_lat=-29.00, sw_lon=-67.53,
    ne_lat=-28.98, ne_lon=-67.45
)

print(f"Parcelas en el sector norte (Famatina): {len(parcelas_norte)}")
for p in parcelas_norte:
    print(f"  {p['nombre']}  —  {p['zona']}")

Parcelas en el sector norte (Famatina): 2
  Viña Famatina  —  Famatina
  Viña Famatina  —  Famatina


In [25]:
dao.cerrar()